### **1. Import the required packages**

In [ ]:
import numpy as np
import pandas as pd

#Machine learning related packages
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

### **2. Read and Explore the data**

In [ ]:
data = pd.read_csv('nba_final.csv')

In [ ]:
data.head() #print the top 5 rows for a quick look at the dataset

,Rk,Player.x,Player_ID,Pos1,Pos2,Age,Tm,G,GS,MP,...,Conference,Role,Fvot,FRank,Pvot,PRank,Mvot,MRank,Score,Play
0,170,A.J. Hammons,hammoaj01,C,NaN,24,DAL,22,0,7.4,...,West,Front,786,123,NaN,NaN,NaN,NaN,83.5,No
1,58,Aaron Brooks,brookaa01,PG,NaN,32,IND,65,0,13.8,...,Est,Back,2474,64,NaN,NaN,NaN,NaN,48.2,No
2,157,Aaron Gordon,gordoaa01,SF,NaN,21,ORL,80,72,28.7,...,Est,Front,22774,29,NaN,NaN,NaN,NaN,40.0,No
3,352,Adreian Payne,paynead01,PF,NaN,25,MIN,18,0,7.5,...,West,Front,861,120,1.0,52.0,NaN,NaN,75.5,No
4,10,Al-Farouq Aminu,aminual01,PF,NaN,26,POR,61,25,29.1,...,West,Front,4971,69,7.0,23.0,NaN,NaN,42.8,No


In [ ]:
data.shape  #tells us the number of rows and columns present in the data

(1408, 45)

In [ ]:
data.dtypes  #prints the datatype of values present in each column

,0
Rk,int64
Player.x,object
Player_ID,object
Pos1,object
Pos2,object
Age,int64
Tm,object
G,int64
GS,int64
MP,float64


In [ ]:
#data['Age'].astype(object)  #always remove the missing values before changing the datatype of any column

In [ ]:
data.isnull().sum(axis = 0)

,0
Rk,0
Player.x,0
Player_ID,0
Pos1,0
Pos2,1396
Age,0
Tm,0
G,0
GS,0
MP,0


In [ ]:
data.isnull().sum(axis = 1) #counts the number of missing values row-wise

,0
0,6
1,5
2,5
3,3
4,3
...,...
1403,1
1404,1
1405,4
1406,1


In [ ]:
data.drop(columns = 'Pos2', inplace = True)

In [ ]:
data = data.fillna(0)

In [ ]:
#check for duplicates
data.duplicated().sum()

np.int64(0)

In [ ]:
#deleting useless columns
data.drop(columns = ['Player.x', 'Player_ID'], inplace = True)

#### **Encoding the categorical columns using LabelEncoder**

In [ ]:
obj_cols = data.select_dtypes(object).columns

In [ ]:
print(obj_cols)

Index(['Pos1', 'Tm', 'Season', 'Conference', 'Role', 'Play'], dtype='object')


In [ ]:
le = LabelEncoder()

for col in obj_cols:
  data[col] = le.fit_transform(data[col])

In [ ]:
data['Play'].value_counts()   #check for data imbalance

,count
Play,
0,1335
1,73


In [ ]:
print(1335 / 1408)
print(73/1408)

0.9481534090909091
0.05184659090909091


### **3. Machine Learning Process**

In [ ]:
X = data.drop(columns = 'Play')
y = data['Play']

In [ ]:
#split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 100, stratify = y)

#### **Scaling or Mean Centering of the data**

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### **Apply Logistic Regression**

In [ ]:
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)

LogisticRegression()

In [ ]:
y_pred = log_reg.predict(X_test_scaled)

In [ ]:
accuracy_score(y_test, y_pred) #don't use the accuracy_score for imbalanced classification data instead use roc_auc_score

0.9787234042553191

In [ ]:
roc_auc_score(y_test, y_pred)

np.float64(0.9258426966292135)

In [ ]:
data.shape

(1408, 42)

### **Now let's the transform the data with PCA**

In [ ]:
pca = PCA(n_components = 0.9)

In [ ]:
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
X_train_pca.shape

(1126, 16)

In [ ]:
pca.explained_variance_ratio_

array([0.37905378, 0.12027188, 0.06486726, 0.06188172, 0.04860752,
       0.03244805, 0.03094522, 0.02541365, 0.02488225, 0.02116037,
       0.01936466, 0.01860224, 0.0163064 , 0.01455815, 0.01320499,
       0.01183306])

In [ ]:
total_explained_variance = sum(pca.explained_variance_ratio_)
print(total_explained_variance)

0.9034012089866098


In [ ]:
log_reg_2 = LogisticRegression()
log_reg_2.fit(X_train_pca, y_train)

LogisticRegression()

In [ ]:
y_pred_2 = log_reg_2.predict(X_test_pca)

In [ ]:
roc_auc_score(y_test, y_pred_2)

np.float64(0.9239700374531835)

### **Now let's apply LDA on this dataset**

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [ ]:
lda = LinearDiscriminantAnalysis()

In [ ]:
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)

#### **min(n_classes - 1, n_features)**

In [ ]:
X_train_lda.shape

(1126, 1)

In [ ]:
log_reg_3 = LogisticRegression()
log_reg_3.fit(X_train_lda, y_train)

LogisticRegression()

In [ ]:
y_pred_3 = log_reg_3.predict(X_test_lda)

In [ ]:
roc_auc_score(y_test, y_pred_3)

np.float64(0.8591760299625467)